# Prepare indexed paired image data for image-image translation model training

## Overview
In this notebook, we will index the image data into paired datasets for image-image translation model training, with the feature being the bright field channels and target being the 5 fluorescent channels, straitfying based on condition (cell line, seeding density, time point).  

## Import libraries

In [1]:
import os
import pathlib
import pandas as pd

In [2]:
# Location tothe extracted metadata information
METADATA_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "metadata"
)
TRAIN_INDEX_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "train_data_index.csv"
)

## Load filtered metadata and channel definition

In [3]:
filtered_metadata = pd.read_csv(os.path.join(METADATA_PATH, 'filtered_metadata.csv'), index_col=None)
channel_mappings = pd.read_csv(os.path.join(METADATA_PATH, 'channel_mappings.csv'), index_col=None)

## Construct an index for each condition pairing the bright field channel image path with the corresponding paths of the fluoresenct channels

In [4]:
# Define the target input channel and the fixed order of target channels
brightfield_channel = "Brightfield high"
target_channel_order = ["Alexa 488", "Alexa 647", "Alexa 568", "Alexa 488 Long (CP)", "HOECHST 33342"]

# List to store data rows
dataset_rows = []

# Group by the stratification variables
stratification_vars = ['PlateID', 'well', 'FieldID', 'Measurement']
grouped_metadata = filtered_metadata.groupby(stratification_vars)

# Process each group
for group_keys, group_data in grouped_metadata:
    plate_id = group_keys[0]
    
    # Get the channel mapping for this PlateID
    if plate_id not in channel_mappings["PlateID"].values:
        continue  # Skip if PlateID is not in channel_mappings
    channel_mapping = channel_mappings[channel_mappings["PlateID"] == plate_id].iloc[0]
    
    # Map ChannelID to channel names
    channel_id_to_name = {
        idx + 1: channel_mapping[f"channel_{idx + 1}"] 
        for idx in range(6)  # Assuming channel_1 to channel_6
    }
    
    # Filter for Brightfield channel
    brightfield_images = group_data[group_data["ChannelID"] == 1]  # Brightfield corresponds to ChannelID 1
    if brightfield_images.empty:
        continue  # Skip if no brightfield images in this group

    # Get paths to brightfield images
    brightfield_path = brightfield_images.iloc[0]["tiff_path"]  # Assuming one brightfield image per group
    
    # Collect paths for the target channels in the specified order
    target_paths = {}
    for target_channel_name in target_channel_order:
        # Find the corresponding ChannelID for the target channel name
        target_channel_id = next(
            (id for id, name in channel_id_to_name.items() if name == target_channel_name), None
        )
        if target_channel_id is not None:
            target_images = group_data[group_data["ChannelID"] == target_channel_id]
            if not target_images.empty:
                target_paths[target_channel_name] = target_images.iloc[0]["tiff_path"]
            else:
                target_paths[target_channel_name] = None  # If no image found for this channel
        else:
            target_paths[target_channel_name] = None  # If channel not defined in mapping
    
    # Ensure all target channels are present; otherwise, skip this group
    if all(path is not None for path in target_paths.values()):
        dataset_rows.append({
            "PlateID": plate_id,
            "well": group_keys[1],
            "FieldID": group_keys[2],
            "Measurement": group_keys[3],
            "Brightfield": brightfield_path,
            **{channel: target_paths[channel] for channel in target_channel_order}
        })

# Convert to DataFrame for saving and inspection
paired_data_df = pd.DataFrame(dataset_rows)

# Log final result
print(f"Paired dataset contains {len(paired_data_df)} rows.")
paired_data_df.head()


Paired dataset contains 10248 rows.


,PlateID,well,FieldID,Measurement,Brightfield,Alexa 488,Alexa 647,Alexa 568,Alexa 488 Long (CP),HOECHST 33342
0,BR00143976,C03,1,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
1,BR00143976,C03,2,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
2,BR00143976,C03,3,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
3,BR00143976,C03,4,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...
4,BR00143976,C03,5,2,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...


## Export index file

In [5]:
paired_data_df.to_csv(TRAIN_INDEX_PATH, index=False)